<a href="https://colab.research.google.com/github/Adhiaris/UAS-FOR-MACHINE-LEARNING-Fraud-Check/blob/main/Task_1_Fraud_Check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q optuna lime mlflow

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer

import optuna
import lime
import lime.lime_tabular
import mlflow
import mlflow.sklearn

import tensorflow as tf
from tensorflow import keras


In [ ]:
import pandas as pd
df = pd.read_csv('midterm-regresi-dataset.csv', header=None)
df.columns = ['target'] + [f'feature_{i}' for i in range(1, df.shape[1])]
df.head()
df.isnull().sum().sum()

In [ ]:
X = df.drop(columns=['target'])
y = df['target']

imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

Q1 = np.percentile(X_imputed, 25, axis=0)
Q3 = np.percentile(X_imputed, 75, axis=0)
IQR = Q3 - Q1
mask = ~((X_imputed < (Q1 - 1.5 * IQR)) | (X_imputed > (Q3 + 1.5 * IQR))).any(axis=1)
X_clean = X_imputed[mask]
y_clean = y.values[mask]

X_train, X_test, y_train, y_test = train_test_split(X_clean, y_clean, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")


## Machine Learning Model

In [ ]:
from sklearn.linear_model import Ridge

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    alpha = trial.suggest_float('alpha', 1e-3, 100.0, log=True)
    model = Ridge(alpha=alpha)
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    return mean_squared_error(y_test, preds)

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

best_alpha = study.best_params['alpha']
print(f"Best alpha: {best_alpha:.4f}")


In [ ]:
mlflow.set_experiment("midterm-regression")

with mlflow.start_run(run_name="Ridge"):
    model = Ridge(alpha=best_alpha)
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)

    mse  = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_test, preds)
    r2   = r2_score(y_test, preds)

    mlflow.log_param("alpha", best_alpha)
    mlflow.log_metric("MSE",  mse)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("MAE",  mae)
    mlflow.log_metric("R2",   r2)
    mlflow.sklearn.log_model(model, "ridge_model")

pd.DataFrame({'Metric': ['MSE', 'RMSE', 'MAE', 'R²'],
              'Value':  [mse, rmse, mae, r2]})


In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(y_test, preds, alpha=0.3, s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted (Ridge Regression)")
plt.tight_layout()
plt.show()


In [ ]:
feature_names = [f'feature_{i}' for i in range(1, X_train_scaled.shape[1]+1)]

explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train_scaled,
    feature_names=feature_names,
    mode='regression'
)

idx = 0
exp = explainer.explain_instance(X_test_scaled[idx], model.predict, num_features=10)
exp.as_pyplot_figure()
plt.title(f"LIME Explanation - Sample {idx} (Actual: {y_test[idx]:.0f}, Pred: {preds[idx]:.0f})")
plt.tight_layout()
plt.show()
